# 17-file-formats-io

17-file-formats-io/01-file-formats-and-io.py

Covers Parquet, CSV, ORC, JSON read/write; partitionBy; save modes;
compression codecs; schema-on-read; coalesce(1) single-file export.

Run:
    python 01-file-formats-and-io.py
Output files land under: pyspark-implementation/data/output/

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# ── SparkSession ──────────────────────────────────────────────────────────────
spark = get_spark("17-file-formats-io")
dfs   = register_views(spark)
emp   = dfs["employees"]
sal   = dfs["salary_history"]


# ── Section 1: Parquet — write and read ──────────────────────────────────────
# Parquet is the preferred columnar format for Spark.
# It preserves schema (types, nullability), supports predicate pushdown,
# and compresses well with Snappy or GZIP.
print("\n" + "="*55)
print("  Section 1: Parquet — write and read")
print("="*55)

parquet_path = output_path("parquet/employees")
emp.write.mode("overwrite").parquet(parquet_path)

df_parquet = spark.read.parquet(parquet_path)
print("Parquet row count:", df_parquet.count())
df_parquet.printSchema()   # all types preserved exactly as written
df_parquet.show(3)


# ── Section 2: Parquet with partitionBy ──────────────────────────────────────
# partitionBy creates subdirectories (dept_id=1/, dept_id=2/, …).
# When reading with a filter on the partition column, Spark skips irrelevant
# directories entirely (partition pruning) — no data read, no shuffle.
print("\n" + "="*55)
print("  Section 2: Parquet with partitionBy (partition pruning)")
print("="*55)

partitioned_path = output_path("parquet/emp_partitioned_by_dept")
emp.write.mode("overwrite").partitionBy("dept_id").parquet(partitioned_path)

# Spark reads only dept_id=1/ — check Spark UI → SQL tab for PartitionFilter
spark.read.parquet(partitioned_path).filter(F.col("dept_id") == 1).show(5)
print("Partition directories:", [d for d in os.listdir(partitioned_path) if d.startswith("dept_id")])


# ── Section 3: CSV — write and read ──────────────────────────────────────────
# CSV stores everything as text; types must be re-inferred or explicitly
# provided on read.  inferSchema scans the file once to guess types.
print("\n" + "="*55)
print("  Section 3: CSV — write and read")
print("="*55)

csv_path = output_path("csv/employees")
emp.write.mode("overwrite").option("header", "true").csv(csv_path)

# Inferred schema — dates often become STRING
df_csv_inferred = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_path)
print("CSV inferred schema:")
df_csv_inferred.printSchema()

# Explicit schema — correct types, no extra scan
from seed_data import EMP_SCHEMA
df_csv_schema = spark.read.option("header", "true").schema(EMP_SCHEMA).csv(csv_path)
print("CSV explicit schema:")
df_csv_schema.printSchema()


# ── Section 4: CSV options — sep, nullValue, dateFormat ──────────────────────
# Real-world CSVs often use non-comma delimiters and custom NULL markers.
# Always match write options to read options or nulls become the string "NULL".
print("\n" + "="*55)
print("  Section 4: CSV options — sep / nullValue / dateFormat")
print("="*55)

csv_path2 = output_path("csv/sal_history")
sal.write.mode("overwrite") \
   .option("header",     "true") \
   .option("sep",        "|") \
   .option("nullValue",  "NULL") \
   .option("dateFormat", "yyyy-MM-dd") \
   .csv(csv_path2)

# emp 10 and 15 have salary=None → written as "NULL" in the file
spark.read \
     .option("header",    "true") \
     .option("sep",       "|") \
     .option("nullValue", "NULL") \
     .csv(csv_path2) \
     .show(3)
# salary_before column will show null correctly because nullValue="NULL" on read


# ── Section 5: ORC — write and read ──────────────────────────────────────────
# ORC is Hive's native columnar format; excellent compression and fast on
# Hive/Presto.  Less common in pure Spark pipelines but supported natively.
print("\n" + "="*55)
print("  Section 5: ORC — write and read")
print("="*55)

orc_path = output_path("orc/salary_history")
sal.write.mode("overwrite").orc(orc_path)
df_orc = spark.read.orc(orc_path)
print("ORC row count:", df_orc.count())
df_orc.show(3)


# ── Section 6: JSON — write and read ─────────────────────────────────────────
# JSON is self-describing but verbose and slow; useful for semi-structured data.
# Spark writes one JSON object per line (JSON Lines / NDJSON format).
# Schema is inferred on read by sampling the file.
print("\n" + "="*55)
print("  Section 6: JSON — write and read")
print("="*55)

json_path = output_path("json/projects")
proj = dfs["projects"]
proj.write.mode("overwrite").json(json_path)
df_json = spark.read.json(json_path)
print("JSON inferred schema:")
df_json.printSchema()   # dates may infer as strings; budget NULL preserved
df_json.show(3)


# ── Section 7: Save modes ────────────────────────────────────────────────────
# overwrite     — replaces existing data entirely
# append        — adds new files; row count grows across writes
# ignore        — silently skips write if path already exists
# errorIfExists — raises AnalysisException if path exists (Spark default)
print("\n" + "="*55)
print("  Section 7: Save modes — overwrite / append")
print("="*55)

# overwrite: start fresh
emp.write.mode("overwrite").parquet(parquet_path)

# append: write 3 rows, then 2 more; total should be 5
append_path = output_path("parquet/emp_append")
emp.limit(3).write.mode("overwrite").parquet(append_path)   # reset to known state
emp.limit(3).write.mode("append").parquet(append_path)      # no-op here (overwrite above)
count_after_3 = spark.read.parquet(append_path).count()
print(f"After first write (3 rows): {count_after_3}")

emp.limit(2).write.mode("append").parquet(append_path)
count_after_5 = spark.read.parquet(append_path).count()
print(f"After append (2 more rows): {count_after_5}")      # expect 5


# ── Section 8: Compression codecs ────────────────────────────────────────────
# Snappy — fast compress/decompress; moderate ratio; default for Parquet
# GZIP   — slower but smaller files; better for cold storage / archiving
# None   — uncompressed; fastest read; largest files
print("\n" + "="*55)
print("  Section 8: Compression codecs — Snappy vs GZIP")
print("="*55)

snappy_path = output_path("parquet/emp_snappy")
gzip_path   = output_path("parquet/emp_gzip")

emp.write.mode("overwrite").option("compression", "snappy").parquet(snappy_path)
emp.write.mode("overwrite").option("compression", "gzip").parquet(gzip_path)

for path, name in [(snappy_path, "snappy"), (gzip_path, "gzip")]:
    total = sum(
        os.path.getsize(os.path.join(path, f))
        for f in os.listdir(path)
        if f.endswith(".parquet") or f.endswith(".gz")
    )
    print(f"{name:8s}: {total / 1024:.1f} KB")
# GZIP file should be smaller; verify in data/output/parquet/


# ── Section 9: Schema-on-read and schema mismatch ────────────────────────────
# Writing a subset of columns, then reading with the full schema:
# columns absent from the file are filled with NULL (schema projection).
# This is schema evolution in action — no error, just nulls.
print("\n" + "="*55)
print("  Section 9: Schema-on-read — missing columns become NULL")
print("="*55)

partial_path = output_path("parquet/sal_partial")
sal.select("emp_id", "salary_after").write.mode("overwrite").parquet(partial_path)

from seed_data import SAL_HIST_SCHEMA
df_partial = spark.read.schema(SAL_HIST_SCHEMA).parquet(partial_path)
df_partial.show(3)
# hist_id, salary_before, effective_date, change_reason, changed_by → all NULL
# Only emp_id and salary_after are populated


# ── Section 10: Single-file export with coalesce(1) ──────────────────────────
# coalesce(1) forces all data into one partition → one output file.
# Convenient for small exports or feeding downstream tools that expect one file.
# WARNING: on large data this creates a bottleneck; avoid in production ETL.
print("\n" + "="*55)
print("  Section 10: coalesce(1) — single-file CSV export")
print("="*55)

single_path = output_path("csv/emp_single_file")
emp.coalesce(1).write.mode("overwrite").option("header", "true").csv(single_path)
csv_files = [f for f in os.listdir(single_path) if f.endswith(".csv")]
print(f"Files in output dir: {csv_files}")   # exactly one .csv file
print(f"Row count from single file: {spark.read.option('header','true').csv(single_path).count()}")


# ── Done ──────────────────────────────────────────────────────────────────────
stop_and_wait(spark)